<a href="https://colab.research.google.com/github/adriansivina-droid/scraper-mef/blob/main/PESCA/Pesca_Loreto_Long.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pesca Loreto · LONG mensual

Convierte el Excel mensual de DIREPRO Loreto (`CONSUMO_<MES>_-_<AÑO>-OPP.xlsx`) en una tabla larga, con el mismo
criterio que `SMN_PESCA_LONG` y `UCA_PESCA_LONG`: una fila por mes, estado y especie, con `VOLUMEN` en toneladas y
`VBP = VOLUMEN × precio base 2007` (miles de soles de 2007). Cada mes se agrega al histórico y, si el mes ya estaba, se
reemplaza. Detalle en `PESCA/pesca.md`.

In [ ]:
# ═══ BLOQUE 1 · PARÁMETROS (revisar cada mes) ═══
# Carpetas en Google Drive (se monta en el Bloque 3)
CARPETA_CRUDOS = "/content/drive/MyDrive/PESCA/LORETO/CRUDOS"        # los .xlsx mensuales de DIREPRO Loreto
RUTA_LONG      = "/content/drive/MyDrive/PESCA/LORETO/LOR_PESCA_LONG.xlsx"  # histórico; se crea si no existe
SUBIR_A_MANO   = False   # True: pide los crudos con un botón en vez de leerlos de CARPETA_CRUDOS

FUENTE = 'DESEMBARQUE'   # 'DESEMBARQUE' (Cuadros 06-08, como UCA) o 'EXTRACCION' (Cuadros 02-04)

# Precio base 2007, en miles de soles por tonelada
PRECIOS = {'FRESCO': 3.27504, 'SALPRESO': 3.01589, 'SECO SALADO': 2.98778}

# Especies del Cuadro Nº 07 (Indicadores del sector pesquero); los zúngaros sin fila propia van a "Zúngaro" y el
# resto a "Otros"
ESPECIES_07 = ['Acarahuazú', 'Arahuana', 'Boquichico', 'Carachama', 'Corvina', 'Chambira', 'Chio Chio', 'Fasaco',
               'Gamitana', 'Mota', 'Insimiracu', 'Lisa', 'Leguía', 'Llambina', 'Maparate', 'Palometa', 'Paco', 'Paña',
               'Paiche', 'Ractacara', 'Sábalo', 'Sardina', 'Shuyo', 'Tucunaré', 'Turushuqui', 'Yahuarachi', 'Yaraqui',
               'Yulilla', 'Zúngaro Achara', 'Zúngaro Cunchimama', 'Zúngaro Doncella', 'Zúngaro Dorado',
               'Zúngaro Manitoa', 'Zúngaro Saltón', 'Zúngaro Tigre']

# Especies del cuadro de producción pesquera (VBP); el resto va a "Otros"
ESPECIES_CUADRO = {'Boquichico': 'Boquichico', 'Llambina': 'Llambina/Palometa', 'Palometa': 'Llambina/Palometa'}

# Nombres del crudo que el Cuadro Nº 07 escribe distinto
CORRECCIONES = {'Manitoa': 'Zúngaro Manitoa', 'Zúngaro Cunchi Mama': 'Zúngaro Cunchimama', 'Leguia': 'Leguía',
                'Insimiracuy': 'Insimiracu', 'Sabalo': 'Sábalo', 'Acarahuazu': 'Acarahuazú'}


In [ ]:
# ═══ BLOQUE 2 · FUNCIONES (no hace falta tocar) ═══
import re, unicodedata
from pathlib import Path
import pandas as pd

MESES = {'ENERO': 1, 'FEBRERO': 2, 'MARZO': 3, 'ABRIL': 4, 'MAYO': 5, 'JUNIO': 6, 'JULIO': 7, 'AGOSTO': 8,
         'SETIEMBRE': 9, 'SEPTIEMBRE': 9, 'OCTUBRE': 10, 'NOVIEMBRE': 11, 'DICIEMBRE': 12}

HOJAS = {'DESEMBARQUE': 'desemb', 'EXTRACCION': 'extrac'}
ESTADOS = [('FRESCO', 'fres'), ('SALPRESO', 'salp'), ('SECO SALADO', 'seco')]

COLUMNAS = ['AÑO', 'MES', 'DEPARTAMENTO', 'FUENTE', 'ESTADO', 'ESPECIE', 'ESPECIE_CORREGIDA', 'ESPECIE_OK',
            'ESPECIE_CUADRO', 'VOLUMEN', 'VBP']


def _clave(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode()
    return re.sub(r'[^a-z]', '', s.lower())


def periodo(xls):
    """Año y mes desde el rótulo 'PERIODO : FEBRERO - 2026' de cualquier hoja."""
    for hoja in xls.sheet_names:
        d = pd.read_excel(xls, hoja, header=None, nrows=15)
        for v in d.values.ravel():
            m = re.search(r'PERIODO\s*:\s*([A-ZÁÉÍÓÚ]+)\s*-\s*(\d{4})', str(v).upper())
            if m and m.group(1) in MESES:
                return int(m.group(2)), MESES[m.group(1)]
    raise ValueError('No se encontró el rótulo PERIODO en ninguna hoja')


def buscar_hoja(xls, fuente, estado):
    pref, clave = HOJAS[fuente], dict(ESTADOS)[estado]
    hallada = [h for h in xls.sheet_names if _clave(h).startswith(pref) and clave in _clave(h)]
    if len(hallada) != 1:
        raise ValueError(f'Hoja de {fuente} {estado}: se esperaba una, se hallaron {hallada}')
    return hallada[0]


def leer_cuadro(xls, hoja):
    """Volumen total (t) por especie de un cuadro 'N° | Especie | localidades | Total'."""
    d = pd.read_excel(xls, hoja, header=None)
    fila = next(i for i, v in d.iloc[:, 1].items() if _clave(v) == 'especie')
    enc = [str(v).strip() for v in d.iloc[fila]]
    col_total = max(i for i, v in enumerate(enc) if _clave(v) == 'total')
    filas = []
    for _, r in d.iloc[fila + 1:].iterrows():
        esp = r.iloc[1]
        if pd.isna(esp) or _clave(esp).startswith(('nota', 'fuente', 'elaboracion')):
            break
        filas.append((str(esp).strip(), pd.to_numeric(r.iloc[col_total], errors='coerce')))
    t = pd.DataFrame(filas, columns=['ESPECIE', 'VOLUMEN']).fillna({'VOLUMEN': 0})
    es_total = t.ESPECIE.map(_clave).eq('total')
    total, t = t.loc[es_total, 'VOLUMEN'].sum(), t[~es_total]
    if abs(t.VOLUMEN.sum() - total) > 0.01:
        print(f'  ⚠ {hoja}: la suma de especies ({t.VOLUMEN.sum():.3f}) no cuadra con el Total ({total:.3f})')
    return t


def corregir(esp):
    e = ' '.join(esp.split()).title()
    return CORRECCIONES.get(e, e)


def especie_ok(esp):
    oficial = {_clave(e): e for e in ESPECIES_07}
    if _clave(esp) in oficial:
        return oficial[_clave(esp)]
    return 'Zúngaro' if _clave(esp).startswith('zungaro') else 'Otros'


def long_loreto(ruta, fuente=FUENTE):
    xls = pd.ExcelFile(ruta)
    anio, mes = periodo(xls)
    partes = []
    for estado, _ in ESTADOS:
        t = leer_cuadro(xls, buscar_hoja(xls, fuente, estado))
        t['ESTADO'] = estado
        partes.append(t)
    df = pd.concat(partes, ignore_index=True)
    df['ESPECIE_CORREGIDA'] = df.ESPECIE.map(corregir)
    df['ESPECIE_OK'] = df.ESPECIE_CORREGIDA.map(especie_ok)
    df['ESPECIE_CUADRO'] = df.ESPECIE_OK.map(ESPECIES_CUADRO).fillna('Otros')
    df['VBP'] = df.VOLUMEN * df.ESTADO.map(PRECIOS)
    df['AÑO'], df['MES'], df['DEPARTAMENTO'] = anio, mes, 'LORETO'
    df['FUENTE'] = 'Desembarque' if fuente == 'DESEMBARQUE' else 'Extracción'
    return df[COLUMNAS]


def unir_historico(nuevo, historico=None):
    """Reemplaza en el histórico los meses que trae lo nuevo; el resto se conserva."""
    if historico is not None and len(historico):
        meses = set(zip(nuevo['AÑO'], nuevo['MES']))
        historico = historico[[(a, m) not in meses for a, m in zip(historico['AÑO'], historico['MES'])]]
        nuevo = pd.concat([historico, nuevo], ignore_index=True)
    orden = {e: i for i, (e, _) in enumerate(ESTADOS)}
    return (nuevo.assign(_o=nuevo.ESTADO.map(orden))
                 .sort_values(['AÑO', 'MES', '_o', 'ESPECIE_CORREGIDA']).drop(columns='_o').reset_index(drop=True))


def cuadro_produccion(df):
    """Cuadro 'Loreto: producción pesquera': VBP, toneladas por estado y por especie, un mes por columna."""
    df = df.assign(PERIODO=df['AÑO'].astype(str) + '-' + df['MES'].astype(str).str.zfill(2))
    vbp = df.pivot_table(columns='PERIODO', values='VBP', aggfunc='sum')
    vbp.index = ['Valor Bruto Pesquero (miles de S/ de 2007)']
    est = df.pivot_table(index='ESTADO', columns='PERIODO', values='VOLUMEN', aggfunc='sum', fill_value=0)
    est = est.reindex(['FRESCO', 'SECO SALADO', 'SALPRESO'], fill_value=0)
    est.index = ['Estado fresco (t)', 'Estado seco salado (t)', 'Estado salpreso (t)']
    esp = df.pivot_table(index='ESPECIE_CUADRO', columns='PERIODO', values='VOLUMEN', aggfunc='sum', fill_value=0)
    esp = esp.reindex(['Boquichico', 'Llambina/Palometa'], fill_value=0)
    esp.index = [f'{e} (t)' for e in esp.index]
    t = pd.concat([vbp, est, esp])
    t.insert(0, 'P=(S/.TMB) 2007', [None] + [PRECIOS[e] * 1000 for e in ['FRESCO', 'SECO SALADO', 'SALPRESO']]
             + [None] * len(esp))
    t.index.name = 'VALOR BRUTO Y PRODUCTOS'
    return t


In [ ]:
# ═══ BLOQUE 3 · GENERAR EL LONG ═══
from google.colab import drive, files
drive.mount('/content/drive')

if SUBIR_A_MANO:
    subidos = files.upload()
    crudos = sorted(Path(n) for n in subidos if n.lower().endswith('.xlsx'))
else:
    crudos = sorted(p for p in Path(CARPETA_CRUDOS).glob('*.xlsx') if not p.name.startswith('~$'))
if not crudos:
    raise FileNotFoundError(f'No hay .xlsx en {CARPETA_CRUDOS}')

nuevos = []
for ruta in crudos:
    print(f'📄 {ruta.name}')
    t = long_loreto(ruta)
    a, m = t['AÑO'].iat[0], t['MES'].iat[0]
    print(f'   {a}-{m:02d} · {len(t)} filas · {t.VOLUMEN.sum():,.3f} t · VBP {t.VBP.sum():,.3f} mil S/ de 2007')
    nuevos.append(t)
nuevo = pd.concat(nuevos, ignore_index=True)
if nuevo.duplicated(['AÑO', 'MES', 'ESTADO', 'ESPECIE']).any():
    print('⚠ Hay dos crudos del mismo mes: se queda el último de la lista')
    nuevo = nuevo.drop_duplicates(['AÑO', 'MES', 'ESTADO', 'ESPECIE'], keep='last')

historico = pd.read_excel(RUTA_LONG, sheet_name='LOR_PESCA') if Path(RUTA_LONG).exists() else None
long_ = unir_historico(nuevo, historico)

Path(RUTA_LONG).parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(RUTA_LONG, engine='openpyxl') as xw:
    long_.to_excel(xw, sheet_name='LOR_PESCA', index=False)
    cuadro_produccion(long_).to_excel(xw, sheet_name='CUADRO')
meses = long_[['AÑO', 'MES']].drop_duplicates()
print(f'✅ {RUTA_LONG}: {len(long_):,} filas, {len(meses)} meses '
      f"({meses.iloc[0, 0]}-{meses.iloc[0, 1]:02d} a {meses.iloc[-1, 0]}-{meses.iloc[-1, 1]:02d})")
display(long_.groupby(['AÑO', 'MES', 'ESTADO'])[['VOLUMEN', 'VBP']].sum().round(3))
